# Solvated ibuprofen

This notebook shows how to set up a simulation cell with two ibuprofen molecules in water, using the same [Open Force Field (OpenFF)](https://openforcefield.org/) toolkit as in [01_ibuprofen_gas_phase.ipynb](01_ibuprofen_gas_phase.ipynb).

In [ ]:
%matplotlib widget
import os
from sys import stdout

import matplotlib.pyplot as plt
import mdtraj
import nglview

# OpenFF is not imported with "from openff... import ..." to avoid name collisions
# with OpenMM, e.g. both packages define a ForceField and a Topology class.
import openff.packmol
import openff.toolkit
import openmm as mm
import pandas
import requests
from openmm import app, unit

## 1. Download molecules from PubChem

As explained in the previous notebook, PubChem usually provides the neutral form of a molecule.
For simplicity, ibuprofen is simulated here as a neutral molecule, even though it would be mostly deprotonated in water at physiological pH.

In [ ]:
# Provide the PubChem compound IDs as strings:
cids = [
    "3672",  # ibuprofen
    "962",  # water
]
# Residue names (at most 3 characters) used in the topology and PDB files.
resnames = [
    "IBU",
    "HOH",
]
# Number of times each molecule is added to the simulation cell.
num_molecules = [2, 500]

In [ ]:
molecules = []
for cid, resname in zip(cids, resnames, strict=True):
    # Download the SDF file if not present yet.
    fn_sdf = f"CID_{cid}.sdf"
    if not os.path.isfile(fn_sdf):
        url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/CID/{cid}/record/SDF/?record_type=3d&response_type=save"
        response = requests.get(url, timeout=60)
        # Stop with an error if the download failed, e.g. for a wrong CID.
        response.raise_for_status()
        with open(fn_sdf, "w") as f:
            f.write(response.text)
    # Load the SDF file as an OpenFF Molecule and set its residue name.
    molecule = openff.toolkit.Molecule.from_file(fn_sdf)
    for atom in molecule.atoms:
        atom.metadata["residue_name"] = resname
    molecules.append(molecule)

## 2. Build the explicit solvent model with Packmol

Packmol is a tool to place molecules at random positions in a simulation cell, without any overlap.
We use it here to generate random mixtures, but it can also be used for more advanced setups.
See the [Packmol website](https://m3g.github.io/packmol/) and the Packmol paper ([10.1002/jcc.21224](https://doi.org/10.1002/jcc.21224)) for more details.

Packmol is called through the [openff-packmol](https://github.com/openforcefield/openff-packmol) wrapper, which takes OpenFF `Molecule` objects as input and returns an OpenFF `Topology`.

In [ ]:
# Run Packmol through the openff-packmol wrapper.
# The size of the cubic box is derived from the total mass and a rough guess of the density.
topology = openff.packmol.pack_box(
    molecules,
    num_molecules,
    target_density=openff.toolkit.Quantity(1.0, "g/mL"),
    box_shape=openff.packmol.UNIT_CUBE,
)
print("Box vectors:", topology.box_vectors)
topology.to_file("packmol_02.pdb")

The initial structure, visualized below, is not yet a realistic liquid: the density is only a rough guess and Packmol keeps atoms of different molecules at least 2 Å apart.
This is not a problem since the system will be equilibrated with NpT molecular dynamics, which will gradually adjust the cell size.

In [ ]:
# Visualize the initial solvated system
traj_init = mdtraj.load("packmol_02.pdb")
view = nglview.show_mdtraj(traj_init)
view.clear_representations()
view.add_licorice()
view.add_unitcell()
view

## 3. Assign Sage parameters with the SMIRNOFF engine

In this section, the force-field parameters are assigned using the methodology developed in the OpenFF community.
The molecules were loaded from SDF files because these contain bond-order data, which are needed to assign the correct parameters.

The Sage 2.3.0 force field also contains parameters for the TIP3P water model, so one force-field file covers all molecules in the simulation cell.
As in the previous notebook, the partial charges of ibuprofen are computed with [NAGL](https://docs.openforcefield.org/projects/nagl/en/stable/).

In [ ]:
force_field = openff.toolkit.ForceField("openff-2.3.0.offxml")
interchange = force_field.create_interchange(topology)
# Convert the result into objects that can be used by OpenMM.
system = interchange.to_openmm_system()
omm_topology = interchange.to_openmm_topology()
positions = interchange.get_positions().to_openmm()

The following cell prints the nonbonded parameters (charge, $\sigma$ and $\epsilon$) of the last atom, which you can use to check whether the right parameters were assigned.

For example, when running this notebook without modifications, the last molecule is water and the last atom is a hydrogen.
You should observe the TIP3P charge of the hydrogen atom, which you can compare to the `q-tip3p-H` parameter in the [force-field file](https://github.com/openforcefield/openff-forcefields/blob/main/openforcefields/offxml/openff-2.3.0.offxml).

In [ ]:
for force in system.getForces():
    if isinstance(force, mm.NonbondedForce):
        npart = force.getNumParticles()
        print(force.getParticleParameters(npart - 1))

## 4. Short NpT molecular dynamics simulation

The code below closely follows the explicit-solvent simulation of alanine dipeptide in [../02_alanine_dipeptide/01_force_fields.ipynb](../02_alanine_dipeptide/01_force_fields.ipynb), with a few differences:

- The `System`, the topology and the initial positions are all taken from the `Interchange` object, instead of from OpenMM's `ForceField` and `Modeller` classes.
- No `nonbondedMethod` or `constraints` arguments are needed: these settings are part of the Sage force field, which uses PME for the electrostatic interactions, constrains the lengths of all bonds involving hydrogen atoms and keeps the water molecules rigid.
- The simulation is ten times shorter: 10000 steps, i.e. 20 picoseconds.

In [ ]:
# Set up the MD simulation.
temperature = 300 * unit.kelvin
pressure = 1 * unit.bar
integrator = mm.LangevinMiddleIntegrator(temperature, 1 / unit.picosecond, 2 * unit.femtoseconds)
system.addForce(mm.MonteCarloBarostat(pressure, temperature))
simulation = app.Simulation(omm_topology, system, integrator)
simulation.context.setPositions(positions)
simulation.minimizeEnergy()

# Write the minimized structure to a PDB file.
# It is used as topology for MDTraj below and can be useful for debugging.
positions = simulation.context.getState(getPositions=True).getPositions()
with open("init_02.pdb", "w") as f:
    app.PDBFile.writeFile(simulation.topology, positions, f)

# Set the reporters collecting the MD output.
simulation.reporters = []
simulation.reporters.append(app.DCDReporter("traj_02.dcd", 100))
simulation.reporters.append(
    app.StateDataReporter(stdout, 1000, step=True, temperature=True, elapsedTime=True)
)
simulation.reporters.append(
    app.StateDataReporter(
        "scalars_02.csv",
        100,
        time=True,
        potentialEnergy=True,
        totalEnergy=True,
        temperature=True,
        volume=True,
    )
)
simulation.step(10000)

In [ ]:
traj = mdtraj.load("traj_02.dcd", top="init_02.pdb")
view = nglview.show_mdtraj(traj)
view.clear_representations()
view.add_licorice()
view.add_unitcell()
view

The temperature and the volume of the simulation cell as a function of time give a first impression of the equilibration.
In this example, the initial guess of the density (1 g/mL) is already close to the equilibrium density of the model.
Hence, the volume changes by only a few percent and fluctuates around 16 nm³ after about ten picoseconds.

In [ ]:
df = pandas.read_csv("scalars_02.csv")
df = df.rename(columns={'#"Time (ps)"': "Time (ps)"})
plt.close("temp")
fig, ax = plt.subplots(num="temp")
df.plot(kind="line", x="Time (ps)", y="Temperature (K)", ax=ax)
plt.close("volume")
fig, ax = plt.subplots(num="volume")
df.plot(kind="line", x="Time (ps)", y="Box Volume (nm^3)", ax=ax);

## 5. Changing compositions

**<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON! &#x1F528;
</span>**

> Try a few different compositions of the simulation cell to verify that you can include practically any molecule you like:
>
> - Remove the ibuprofen molecules.
> - Replace the second ibuprofen by another drug molecule (e.g. aspirin).
> - Replace the two ibuprofens by one large drug molecule (e.g. amoxicillin).
> - Replace the solvent (e.g. dimethyl sulfoxide, DMSO).